# Rocket Fast-DeePC Example

Fixed benchmark: $T=225$, initial condition $(0.65, 0.92, 0.05)$. This notebook times the controller formulations and records the original and C Fast-DeePC landings.Please use Python 3.10 to run it


## 1. Import packages and locate the repository


In [ ]:
from pathlib import Path
import ctypes as ct
import os
import random
import sys
import time
import warnings

# diffcp/cvxpylayers require the MSYS2 runtime DLLs on this machine.
_msys2_dll_directory = os.add_dll_directory(r"C:\msys64\ucrt64\bin")

import cvxpy as cp
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import osqp
import scipy.sparse as sp
from scipy.linalg import cho_factor, cho_solve

import coco_rocket_lander  # Registers coco_rocket_lander/RocketLander-v0.
from deepc_hunt.controllers import npDeePC

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.set_printoptions(precision=6, suppress=True)

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
PROJECT_DIR = ROOT
ROCKET_REPO = Path(os.environ.get("DEEPC_HUNT_DIR", ROOT / "external" / "DeePC-Hunt-main"))
DATA_DIR = ROCKET_REPO / "examples" / "data"

DLL_CANDIDATES = [
    ROOT / "build" / "deepc_boxqp_optimized.dll",
]
DLL_PATH = next((path for path in DLL_CANDIDATES if path.exists()), None)
if DLL_PATH is None:
    raise FileNotFoundError("deepc_boxqp_optimized.dll was not found")

print("Python environment ready")
print("CVXPY:", cp.__version__)
print("C solver:", DLL_PATH)


## 2. Define the fixed rocket, DeePC, and constraint settings


In [ ]:
# Shared DeePC and rocket settings.
T_INI = 1
HORIZON = 10
DIM_U = 3
DIM_Y = 6
NU = HORIZON * DIM_U
NY = HORIZON * DIM_Y
NZ = NU + NY
Q_STEP = np.diag([100.0, 15.0, 25.0, 2.2, 6000.0, 120.0])
R_STEP = np.diag([0.01, 0.30, 0.30])
WY = 2.0 * np.kron(np.eye(HORIZON), Q_STEP)
WU = 2.0 * np.kron(np.eye(HORIZON), R_STEP)

# Preserve the exact paper weights for the original DeePC-Hunt baseline.
ORIGINAL_Q_STEP = np.diag([100.0, 10.0, 5.0, 1.0, 3000.0, 30.0])
ORIGINAL_R_STEP = np.diag([0.01, 0.01, 0.01])

# Quadratic Fast-DeePC penalties learned in the video example.
LAMBDA_PEN = 1.0e5
RHO_PEN = 1.0e5
LAMBDA_G = 281.1706625951745

# Exact trained parameters from examples/reproduce_paper/rocket.py.
ORIGINAL_PARAMS = {"lam_g1": 50.0,"lam_g2": 10.3,"lam_y": 1000.0,}

U_LOWER_STEP = np.array([0.0, -1.0, -1.0])
U_UPPER_STEP = np.array([1.0, 1.0, 1.0])
PHYSICAL_Y_LOWER_STEP = np.array([0.0, 7.0, -100.0, -100.0, -0.6, -100.0])
PHYSICAL_Y_UPPER_STEP = np.array([33.0, 26.6, 100.0, 100.0, 0.6, 100.0])
# A 0.15-rad prediction margin protects the nonlinear plant's physical pitch box. The original controller continues to use the physical +/-0.6 bound.
Y_LOWER_STEP = np.array([0.0, 7.0, -100.0, -100.0, -0.45, -100.0])
Y_UPPER_STEP = np.array([33.0, 26.6, 100.0, 100.0, 0.45, 100.0])

U_LOWER = np.tile(U_LOWER_STEP, HORIZON)
U_UPPER = np.tile(U_UPPER_STEP, HORIZON)
Y_LOWER = np.tile(Y_LOWER_STEP, HORIZON)
Y_UPPER = np.tile(Y_UPPER_STEP, HORIZON)
ORIGINAL_Y_LOWER = np.tile(PHYSICAL_Y_LOWER_STEP, HORIZON)
ORIGINAL_Y_UPPER = np.tile(PHYSICAL_Y_UPPER_STEP, HORIZON)
Z_LOWER = np.concatenate([U_LOWER, Y_LOWER])
Z_UPPER = np.concatenate([U_UPPER, Y_UPPER])
U_REF = np.zeros(NU)

# Same-problem CVXPY/C accuracy settings.
SOLVER_TOL = 1.0e-5
MAX_SOLVER_ITER = 100000
C_MAX_ITER = 100

# Original problem is unchanged. These are solver-only runtime settings.
ORIGINAL_SOLVER_TOL = 1.0e-4
ORIGINAL_MAX_ITER = 10000
ORIGINAL_POLISHING = False

# One fixed, reproducible landing experiment.
FIXED_INITIAL_POSITION = (0.65, 0.92, 0.05)
FIXED_ENV_SEED = 42
BENCHMARK_SEEDS = [FIXED_ENV_SEED]
all_seeds = BENCHMARK_SEEDS
MAX_STEPS = 600
LOCKSTEP_MAX_STEPS = 80
RUN_FULL_BENCHMARK = False
ENABLE_ORIGINAL_BASELINE = True



## 3. Load the fixed $T=225$ offline trajectory and form the Hankel matrices


In [ ]:
def block_hankel(data, order):
    data = np.asarray(data, dtype=np.float64)
    samples, channels = data.shape
    ncol = samples - order + 1
    if ncol <= 0:
        raise ValueError(f"Need T >= {order}; received T={samples}")
    H = np.empty((order * channels, ncol), dtype=np.float64)
    for column in range(ncol):
        H[:, column] = data[column:column + order].reshape(-1)
    return H


from pathlib import Path

DATA_DIR = Path(
    r"C:\Users\Terry\Desktop\JHU researh\DEEPC\DeePC-Hunt-main\examples\data"
)

u_data = np.loadtxt(
    DATA_DIR / "rocket_ud.csv",
    delimiter=",",
    dtype=np.float64,
)

y_data = np.loadtxt(
    DATA_DIR / "rocket_yd.csv",
    delimiter=",",
    dtype=np.float64,
)
assert u_data.shape[1] == DIM_U
assert y_data.shape[1] == DIM_Y
assert len(u_data) == len(y_data)
assert len(u_data) == 225, "This notebook intentionally keeps T=225"

L = T_INI + HORIZON
HU = block_hankel(u_data, L)
HY = block_hankel(y_data, L)
UP = HU[:T_INI * DIM_U]
UF = HU[T_INI * DIM_U:]
YP = HY[:T_INI * DIM_Y]
YF = HY[T_INI * DIM_Y:]
Z = np.vstack([UP, YP, UF])
NCOL = UF.shape[1]

assert UF.shape == (NU, NCOL)
assert YF.shape == (NY, NCOL)
print("Data length T:", len(u_data))
print("g dimension:", NCOL)
print("Reduced online dimension N(m+p):", NZ)
print("Hankel block shapes:", UP.shape, YP.shape, UF.shape, YF.shape)

def make_environment(seed, initial_position=FIXED_INITIAL_POSITION):
    env = gym.make(
        "coco_rocket_lander/RocketLander-v0",
        args={"initial_position": initial_position},
    )
    observation, info = env.reset(seed=seed)
    return env, np.asarray(observation, dtype=np.float64), info


def reference_from_landing(landing_position):
    step_ref = np.array([
        landing_position[0],
        landing_position[1],
        0.0,
        0.0,
        landing_position[2],
        0.0,
    ], dtype=np.float64)
    return np.tile(step_ref, HORIZON)


def update_window(u_window, y_window, action, output_before_action):
    u_next = np.vstack([u_window[1:], np.asarray(action).reshape(1, DIM_U)])
    y_next = np.vstack([
        y_window[1:],
        np.asarray(output_before_action).reshape(1, DIM_Y),
    ])
    return u_next, y_next


def max_plan_violation(u, y):
    z = np.concatenate([np.asarray(u).reshape(-1), np.asarray(y).reshape(-1)])
    return float(max(0.0, np.max(Z_LOWER - z), np.max(z - Z_UPPER)))


def actual_output_violation(y):
    y = np.asarray(y)[:DIM_Y]
    return float(max(0.0, np.max(PHYSICAL_Y_LOWER_STEP - y), np.max(y - PHYSICAL_Y_UPPER_STEP)))


def full_quadratic_objective(g, u, y, u_ini, y_ini, y_ref):
    u = np.asarray(u).reshape(-1)
    y = np.asarray(y).reshape(-1)
    g = np.asarray(g).reshape(-1)
    w = np.concatenate([np.asarray(u_ini).reshape(-1), np.asarray(y_ini).reshape(-1), u])
    du = u - U_REF
    dy = y - y_ref
    return float(
        0.5 * du @ WU @ du
        + 0.5 * dy @ WY @ dy
        + 0.5 * LAMBDA_PEN * np.sum((Z @ g - w) ** 2)
        + 0.5 * RHO_PEN * np.sum((YF @ g - y) ** 2)
        + 0.5 * LAMBDA_G * np.sum(g ** 2)
    )


def timing_stats(values):
    values = np.asarray(values, dtype=np.float64)
    if values.size == 0:
        return {"mean_ms": np.nan, "median_ms": np.nan, "p95_ms": np.nan, "total_s": 0.0}
    return {
        "mean_ms": 1e3 * float(np.mean(values)),
        "median_ms": 1e3 * float(np.median(values)),
        "p95_ms": 1e3 * float(np.percentile(values, 95)),
        "total_s": float(np.sum(values)),
    }


## 4. Build the full soft-L2 CVXPY baseline


In [ ]:
class FullQuadraticCVXPY:
    name = "Full quadratic CVXPY"

    def __init__(self):
        start = time.perf_counter()
        self.g = cp.Variable(NCOL)
        self.u = cp.Variable(NU)
        self.y = cp.Variable(NY)
        self.u_ini = cp.Parameter(T_INI * DIM_U)
        self.y_ini = cp.Parameter(T_INI * DIM_Y)
        self.y_ref = cp.Parameter(NY)

        target = cp.hstack([self.u_ini, self.y_ini, self.u])
        objective = 0.5 * cp.sum(cp.multiply(np.diag(WU), cp.square(self.u - U_REF)))
        objective += 0.5 * cp.sum(cp.multiply(np.diag(WY), cp.square(self.y - self.y_ref)))
        objective += 0.5 * LAMBDA_PEN * cp.sum_squares(Z @ self.g - target)
        objective += 0.5 * RHO_PEN * cp.sum_squares(YF @ self.g - self.y)
        objective += 0.5 * LAMBDA_G * cp.sum_squares(self.g)

        constraints = [
            self.u >= U_LOWER,
            self.u <= U_UPPER,
            self.y >= Y_LOWER,
            self.y <= Y_UPPER,
        ]
        self.problem = cp.Problem(cp.Minimize(objective), constraints)
        if not self.problem.is_qp():
            raise RuntimeError("The full quadratic baseline was not recognized as a QP")
        self.offline_time = time.perf_counter() - start

    def reset(self):
        for variable in (self.g, self.u, self.y):
            variable.value = None

    def solve(self, u_ini, y_ini, y_ref):
        start = time.perf_counter()
        self.u_ini.value = np.asarray(u_ini).reshape(-1)
        self.y_ini.value = np.asarray(y_ini).reshape(-1)
        self.y_ref.value = np.asarray(y_ref).reshape(-1)
        self.problem.solve(
            solver=cp.OSQP,
            warm_start=True,
            eps_abs=SOLVER_TOL,
            eps_rel=SOLVER_TOL,
            max_iter=MAX_SOLVER_ITER,
            polishing=False,
            adaptive_rho=True,
            check_termination=25,
            verbose=False,
        )
        online_time = time.perf_counter() - start
        if self.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"Full CVXPY status: {self.problem.status}")
        return {
            "u": np.asarray(self.u.value, dtype=np.float64).reshape(-1).copy(),
            "y": np.asarray(self.y.value, dtype=np.float64).reshape(-1).copy(),
            "g": np.asarray(self.g.value, dtype=np.float64).reshape(-1).copy(),
            "objective": float(self.problem.value),
            "online_time": online_time,
            "solver_time": float(self.problem.solver_stats.solve_time or 0.0),
            "status": self.problem.status,
            "iterations": int(self.problem.solver_stats.num_iters or 0),
        }



## 5. Build the C Fast-DeePC BoxQP controller


In [ ]:
BINT = ct.c_long


class BoxQPInfo(ct.Structure):
    _fields_ = [
        ("iterations", BINT),
        ("mu", ct.c_double),
        ("duality", ct.c_double),
        ("status", ct.c_int),
        ("chol_info", BINT),
    ]


class DeePCKForm(ct.Structure):
    _fields_ = [
        ("nfixed", BINT), ("nz", BINT), ("ncol", BINT), ("nrows", BINT),
        ("ridge", ct.c_double),
        ("A", ct.POINTER(ct.c_double)), ("Gchol", ct.POINTER(ct.c_double)),
        ("Hphys", ct.POINTER(ct.c_double)), ("Fphys", ct.POINTER(ct.c_double)),
        ("Hbox", ct.POINTER(ct.c_double)), ("Fbox", ct.POINTER(ct.c_double)),
        ("center", ct.POINTER(ct.c_double)), ("scale", ct.POINTER(ct.c_double)),
        ("hcenter", ct.POINTER(ct.c_double)), ("hbox", ct.POINTER(ct.c_double)),
        ("swarm", ct.POINTER(ct.c_double)), ("sout", ct.POINTER(ct.c_double)),
    ]


lib = ct.CDLL(str(DLL_PATH))
double_p = ct.POINTER(ct.c_double)
lib.deepc_kform_offline.argtypes = [
    ct.POINTER(DeePCKForm), double_p, double_p, BINT, BINT, BINT,
    double_p, double_p, ct.c_double, double_p, double_p, double_p,
]
lib.deepc_kform_offline.restype = ct.c_int
lib.deepc_kform_online_step.argtypes = [
    ct.POINTER(DeePCKForm), double_p, double_p, double_p,
    ct.c_double, BINT, double_p, ct.POINTER(BoxQPInfo),
]
lib.deepc_kform_online_step.restype = ct.c_int
lib.deepc_kform_recover_g.argtypes = [
    ct.POINTER(DeePCKForm), double_p, double_p, double_p,
]
lib.deepc_kform_recover_g.restype = ct.c_int


def ptr(array):
    return array.ctypes.data_as(double_p)


def fmat(array):
    return np.asfortranarray(array, dtype=np.float64)


class ReducedCBoxQP:
    name = "Reduced C BoxQP"

    def __init__(self):
        start = time.perf_counter()
        A_fixed = fmat(np.vstack([UP, YP]))
        A_decision = fmat(np.vstack([UF, YF]))
        fixed_weights = np.full(A_fixed.shape[0], LAMBDA_PEN)
        decision_weights = np.r_[
            np.full(NU, LAMBDA_PEN), np.full(NY, RHO_PEN)
        ]
        decision_cost = fmat(np.block([
            [WU, np.zeros((NU, NY))],
            [np.zeros((NY, NU)), WY],
        ]))
        self.data = DeePCKForm()
        status = lib.deepc_kform_offline(
            ct.byref(self.data), ptr(A_fixed), ptr(A_decision),
            BINT(A_fixed.shape[0]), BINT(NZ), BINT(NCOL),
            ptr(fixed_weights), ptr(decision_weights), ct.c_double(LAMBDA_G),
            ptr(decision_cost), ptr(Z_LOWER), ptr(Z_UPPER),
        )
        if status != 0:
            raise RuntimeError(f"deepc_kform_offline failed with status {status}")
        self.z_warm = None
        self.offline_time = time.perf_counter() - start

    def reset(self):
        self.z_warm = None

    def recover_g(self, fixed_data, u, y):
        z = np.ascontiguousarray(np.r_[u, y], dtype=np.float64)
        g = np.zeros(NCOL, dtype=np.float64)
        fixed_data = np.ascontiguousarray(fixed_data, dtype=np.float64)
        status = lib.deepc_kform_recover_g(
            ct.byref(self.data), ptr(fixed_data), ptr(z), ptr(g)
        )
        if status != 0:
            raise RuntimeError(f"deepc_kform_recover_g failed with status {status}")
        return g

    def solve(self, u_ini, y_ini, y_ref):
        u_ini = np.asarray(u_ini, dtype=np.float64).reshape(-1)
        y_ini = np.asarray(y_ini, dtype=np.float64).reshape(-1)
        y_ref = np.asarray(y_ref, dtype=np.float64).reshape(-1)

        start = time.perf_counter()
        ini = np.ascontiguousarray(np.r_[u_ini, y_ini], dtype=np.float64)
        tracking = np.ascontiguousarray(np.r_[-WU @ U_REF, -WY @ y_ref])
        z = np.zeros(NZ, dtype=np.float64)
        info = BoxQPInfo()
        warm = None if self.z_warm is None else ptr(self.z_warm)
        status = lib.deepc_kform_online_step(
            ct.byref(self.data), ptr(ini), ptr(tracking), warm,
            ct.c_double(SOLVER_TOL),
            BINT(C_MAX_ITER),
            ptr(z),
            ct.byref(info),
        )
        online_time = time.perf_counter() - start
        if status not in (0, 1):
            raise RuntimeError(f"C status={status}, chol_info={info.chol_info}")

        u = z[:NU]
        y = z[NU:]
        self.z_warm = np.ascontiguousarray(z.copy())
        g = self.recover_g(ini, u, y)
        objective = full_quadratic_objective(g, u, y, u_ini, y_ini, y_ref)
        return {
            "u": u.copy(),
            "y": y.copy(),
            "g": g.copy(),
            "objective": objective,
            "online_time": online_time,
            "solver_time": online_time,
            "status": int(status),
            "iterations": int(info.iterations),
            "mu": float(info.mu),
        }



## 6. Build the original hard-L1 DeePC-Hunt baseline and warm up all three controllers


In [ ]:
class OriginalDeePCHunt:
    name = "Original DeePC-Hunt"

    def __init__(self):
        start = time.perf_counter()
        self.controller = npDeePC(
            ud=u_data,
            yd=y_data,
            u_constraints=(U_LOWER, U_UPPER),
            y_constraints=(ORIGINAL_Y_LOWER, ORIGINAL_Y_UPPER),
            Tini=T_INI,
            N=HORIZON,
            m=DIM_U,
            p=DIM_Y,
            n=DIM_Y,
        )
        self.controller.setup(
            Q=ORIGINAL_Q_STEP,
            R=ORIGINAL_R_STEP,
            lam_g1=ORIGINAL_PARAMS["lam_g1"],
            lam_g2=ORIGINAL_PARAMS["lam_g2"],
            lam_y=ORIGINAL_PARAMS["lam_y"],
        )
        if not self.controller.problem.is_qp():
            raise RuntimeError("Original DeePC-Hunt problem is not a QP")
        self.offline_time = time.perf_counter() - start

    def reset(self):
        for variable in self.controller.problem.variables():
            variable.value = None

    def solve(self, u_ini, y_ini, y_ref):
        start = time.perf_counter()
        self.controller.u_ini.value = np.asarray(u_ini).reshape(-1)
        self.controller.y_ini.value = np.asarray(y_ini).reshape(-1)
        self.controller.y_ref.value = np.asarray(y_ref).reshape(-1)
        self.controller.u_ref.value = U_REF
        self.controller.problem.solve(
            solver=cp.OSQP,
            warm_start=True,
            eps_abs=ORIGINAL_SOLVER_TOL,
            eps_rel=ORIGINAL_SOLVER_TOL,
            max_iter=ORIGINAL_MAX_ITER,
            polishing=ORIGINAL_POLISHING,
            adaptive_rho=True,
            check_termination=25,
            verbose=False,
        )
        online_time = time.perf_counter() - start
        if self.controller.problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            raise RuntimeError(f"Original DeePC-Hunt status: {self.controller.problem.status}")
        return {
            "u": np.asarray(self.controller.u.value, dtype=np.float64).reshape(-1).copy(),
            "y": np.asarray(self.controller.y.value, dtype=np.float64).reshape(-1).copy(),
            "g": np.asarray(self.controller.g.value, dtype=np.float64).reshape(-1).copy(),
            "objective": float(self.controller.problem.value),
            "online_time": online_time,
            "solver_time": float(self.controller.problem.solver_stats.solve_time or 0.0),
            "status": self.controller.problem.status,
            "iterations": int(self.controller.problem.solver_stats.num_iters or 0),
        }

cvx_controller = FullQuadraticCVXPY()
c_controller = ReducedCBoxQP()
original_controller = OriginalDeePCHunt() if ENABLE_ORIGINAL_BASELINE else None

warm_seed = all_seeds[0]
warm_env, warm_obs, _ = make_environment(warm_seed)
warm_landing = np.asarray(warm_env.get_landing_position(), dtype=np.float64)
warm_ref = reference_from_landing(warm_landing)
warm_u_ini = np.zeros(T_INI * DIM_U)
warm_y_ini = np.tile(warm_obs[:DIM_Y], T_INI)
warm_env.close()

warmup_times = {}
for controller in [cvx_controller, c_controller, original_controller]:
    if controller is None:
        continue
    start = time.perf_counter()
    result = controller.solve(warm_u_ini, warm_y_ini, warm_ref)
    warmup_times[controller.name] = time.perf_counter() - start
    controller.reset()

print("Offline construction times (s):")
for controller in [cvx_controller, c_controller, original_controller]:
    if controller is not None:
        print(f"  {controller.name}: {controller.offline_time:.6f}")
print("Untimed warm-up solve times (s):", warmup_times)


## 7. Solve the one fixed landing case


In [ ]:
# Run both controllers from the fixed paper initial condition.

import numpy as np
import gymnasium as gym
from pathlib import Path

VIDEO_NOTEBOOK_DIR = ROOT
VIDEO_DIR = ROOT / "media" / "generated_rocket_videos"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_SEED = 42
SEARCH_MAX_STEPS = 650


def make_environment_at(seed, initial_position, render_mode=None, record_video=False, video_subdir=None, name_prefix="rocket"):
    kwargs = {
        "args": {
            "initial_position": tuple(initial_position),
            "enable_wind": False,
        }
    }
    if render_mode is not None:
        kwargs["render_mode"] = render_mode
    env = gym.make("coco_rocket_lander/RocketLander-v0", **kwargs)
    if record_video:
        assert video_subdir is not None
        video_subdir.mkdir(parents=True, exist_ok=True)
        env = gym.wrappers.RecordVideo(
            env,
            video_folder=str(video_subdir),
            episode_trigger=lambda episode_id: True,
            name_prefix=name_prefix,
        )
    obs, info = env.reset(seed=seed)
    return env, np.asarray(obs, dtype=np.float64), info


def run_episode_at_initial(controller, seed, initial_position, max_steps, record_video=False, video_name=None):
    video_subdir = VIDEO_DIR / video_name if record_video else None
    env, obs, _ = make_environment_at(
        seed=seed,
        initial_position=initial_position,
        render_mode="rgb_array" if record_video else None,
        record_video=record_video,
        video_subdir=video_subdir,
        name_prefix=video_name or controller.name.replace(" ", "_"),
    )
    landing = np.asarray(env.get_landing_position(), dtype=np.float64)
    y_ref = reference_from_landing(landing)
    reference_step = y_ref[:DIM_Y]
    controller.reset()

    u_window = np.zeros((T_INI, DIM_U), dtype=np.float64)
    y_window = np.tile(obs[:DIM_Y], (T_INI, 1))
    states = [obs[:DIM_Y].copy()]
    actions = []
    solve_times = []
    iterations = []
    failure_message = None
    touched_ground = False
    terminated_or_truncated = False

    for _ in range(max_steps):
        output_before_action = obs[:DIM_Y].copy()
        if (bool(obs[6]) and bool(obs[7])) or touched_ground:
            action = np.zeros(DIM_U)
            touched_ground = True
        else:
            try:
                result = controller.solve(
                    u_window.reshape(-1),
                    y_window.reshape(-1),
                    y_ref,
                )
                action = result["u"][:DIM_U].copy()
                solve_times.append(result.get("online_time", np.nan))
                iterations.append(result.get("iterations", np.nan))
            except Exception as exc:
                failure_message = repr(exc)
                break

        next_obs, reward, terminated, truncated, info = env.step(action)
        obs = np.asarray(next_obs, dtype=np.float64)
        actions.append(action)
        states.append(obs[:DIM_Y].copy())
        u_window, y_window = update_window(
            u_window,
            y_window,
            action,
            output_before_action,
        )
        terminated_or_truncated = bool(terminated or truncated)
        if terminated_or_truncated:
            break

    env.close()

    success = bool(obs[6]) and bool(obs[7]) and obs[1] >= landing[1]
    final_position_error = float(np.linalg.norm(obs[:2] - landing[:2]))
    final_attitude_error = float(abs(obs[4] - landing[2]))
    return {
        "controller": controller.name,
        "initial_position": tuple(initial_position),
        "landing": landing,
        "states": np.asarray(states),
        "actions": np.asarray(actions),
        "solve_times": np.asarray(solve_times),
        "iterations": np.asarray(iterations),
        "success": bool(success),
        "episode_length": len(actions),
        "final_position_error": final_position_error,
        "final_attitude_error": final_attitude_error,
        "failure_message": failure_message,
        "video_dir": str(video_subdir) if video_subdir is not None else None,
    }




In [ ]:
selected_video_case = {
    "initial": FIXED_INITIAL_POSITION,
    "original": run_episode_at_initial(original_controller, FIXED_ENV_SEED, FIXED_INITIAL_POSITION, MAX_STEPS),
    "c": run_episode_at_initial(c_controller, FIXED_ENV_SEED, FIXED_INITIAL_POSITION, MAX_STEPS),
}
orig_plot_ep = selected_video_case["original"]
c_plot_ep = selected_video_case["c"]
print("Fixed initial condition:", FIXED_INITIAL_POSITION)


## 7. Trojectory

In [ ]:
# Exact two-panel trajectories for the solved video case.

from matplotlib.collections import LineCollection
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib.pyplot as plt


def _rocket_inverted_y_segments(states, number=10, size=0.90):
    """Create long-body, short-arm upside-down-Y rocket symbols."""
    states = np.asarray(states, dtype=np.float64)
    indices = np.unique(
        np.linspace(0, len(states) - 1, min(number, len(states))).astype(int)
    )

    segments = []
    for index in indices:
        x_pos, y_pos = states[index, :2]
        theta = states[index, 4] if states.shape[1] > 4 else 0.0

        body_axis = np.array([-np.sin(theta), np.cos(theta)])
        side_axis = np.array([np.cos(theta), np.sin(theta)])
        tail_joint = np.array([x_pos, y_pos])

        nose = tail_joint + 1.35 * size * body_axis
        left_arm = (
            tail_joint
            - 0.16 * size * body_axis
            - 0.22 * size * side_axis
        )
        right_arm = (
            tail_joint
            - 0.16 * size * body_axis
            + 0.22 * size * side_axis
        )

        segments.extend([
            np.vstack([tail_joint, nose]),
            np.vstack([tail_joint, left_arm]),
            np.vstack([tail_joint, right_arm]),
        ])

    return segments


orig_plot_ep = selected_video_case["original"]
c_plot_ep = selected_video_case["c"]

landing = np.asarray(c_plot_ep["landing"], dtype=np.float64)
all_states = np.vstack([
    orig_plot_ep["states"],
    c_plot_ep["states"],
])

barge_width = (1000.0 / 30.0 / 16.0) * 4.0

x_min = min(13.0, float(np.min(all_states[:, 0])) - 0.45)
x_max = max(21.0, float(np.max(all_states[:, 0])) + 0.45)
y_min = min(6.0, float(landing[1]) - 0.8)
y_max = float(np.max(all_states[:, 1])) + 0.95

with plt.rc_context({
    "font.size": 19,
    "axes.titlesize": 25,
    "axes.labelsize": 22,
    "xtick.labelsize": 17,
    "ytick.labelsize": 17,
    "legend.fontsize": 16,
}):
    fig, axes = plt.subplots(
        1, 2,
        figsize=(16, 7.0),
        sharex=True,
        sharey=True,
    )

    fig.subplots_adjust(
        left=0.08,
        right=0.985,
        bottom=0.16,
        top=0.84,
        wspace=0.15,
    )

    for axis, title, episode, color in [
        (axes[0], "Original DeePC-Hunt", orig_plot_ep, "tab:green"),
        (axes[1], "Fast-DeePC", c_plot_ep, "tab:orange"),
    ]:
        states = np.asarray(episode["states"], dtype=np.float64)

        axis.add_patch(
            Rectangle(
                (landing[0] - barge_width / 2, landing[1] - 0.16),
                barge_width,
                0.16,
                facecolor="#237a3b",
                edgecolor="#12471f",
                linewidth=2.6,
                alpha=0.95,
                label="Landing zone",
                zorder=1,
            )
        )

        axis.plot(
            states[:, 0],
            states[:, 1],
            color=color,
            linewidth=4.2,
            label="Trajectory",
            zorder=2,
        )

        rocket_segments = _rocket_inverted_y_segments(
            states,
            number=10,
            size=0.90,
        )

        axis.add_collection(
            LineCollection(
                rocket_segments,
                colors="white",
                linewidths=9.5,
                alpha=0.98,
                zorder=3,
            )
        )
        axis.add_collection(
            LineCollection(
                rocket_segments,
                colors="#8b0000",
                linewidths=5.0,
                alpha=0.97,
                zorder=4,
            )
        )

        axis.scatter(
            states[0, 0],
            states[0, 1],
            marker="o",
            s=125,
            facecolors="white",
            edgecolors="black",
            linewidths=2.0,
            label="Initial position",
            zorder=7,
        )

        axis.scatter(
            states[-1, 0],
            states[-1, 1],
            marker="o",
            s=95,
            facecolors=color,
            edgecolors="black",
            linewidths=1.4,
            label="Final position",
            zorder=7,
        )

        axis.set_title(title, pad=12)
        axis.set_xlabel("x position (m)", labelpad=12)
        axis.set_xlim(x_min, x_max)
        axis.set_ylim(y_min, y_max)
        axis.grid(True, alpha=0.32, linewidth=1.1)

        legend = axis.legend(
            loc="upper left",
            fontsize=20,
            frameon=True,
            framealpha=0.95,
            facecolor="white",
            edgecolor="gray",
            borderpad=1.0,
            labelspacing=0.8,
            handlelength=2.8,
            handletextpad=0.8,
        )
        legend.get_frame().set_linewidth(1.4)

        for spine in axis.spines.values():
            spine.set_linewidth(1.8)

        axis.tick_params(
            axis="both",
            which="major",
            width=1.7,
            length=7,
        )

    axes[0].set_ylabel("y position (m)", labelpad=12)
    axes[1].set_ylabel("y position (m)", labelpad=1)
    axes[1].tick_params(labelleft=True)

    fig.suptitle(
        "Controller comparison: original DeePC-Hunt and Fast-DeePC",
        fontsize=27,
    )

    plot_path = (
        VIDEO_NOTEBOOK_DIR
        / "rocket_hunt_video_case_trajectory.png"
    )

    plt.savefig(
        plot_path,
        dpi=300,
        bbox_inches="tight",
        pad_inches=0.25,
    )
    plt.show()

print("Saved", plot_path)

## 8. Replay the solved actions and save the two landing videos


In [ ]:
# Record videos by replaying the exact actions used in the preceding trajectory plot.

def replay_actions_to_video(solved_episode, video_name):
    video_subdir = VIDEO_DIR / video_name
    env, observation, _ = make_environment_at(
        seed=SEARCH_SEED,
        initial_position=selected_video_case["initial"],
        render_mode="rgb_array",
        record_video=True,
        video_subdir=video_subdir,
        name_prefix=video_name,
    )

    replayed_states = [observation[:DIM_Y].copy()]
    for action in solved_episode["actions"]:
        observation, _, terminated, truncated, _ = env.step(action)
        observation = np.asarray(observation, dtype=np.float64)
        replayed_states.append(observation[:DIM_Y].copy())
        if terminated or truncated:
            break

    env.close()
    replayed_states = np.asarray(replayed_states, dtype=np.float64)
    solved_states = np.asarray(solved_episode["states"], dtype=np.float64)

    if replayed_states.shape != solved_states.shape:
        raise RuntimeError(
            f"Video replay shape {replayed_states.shape} does not match solved trajectory {solved_states.shape}."
        )

    maximum_state_error = float(np.max(np.abs(replayed_states - solved_states)))
    if maximum_state_error > 1.0e-7:
        raise RuntimeError(
            f"Video replay differs from plotted trajectory: maximum state error={maximum_state_error:.3e}."
        )

    mp4_files = sorted(video_subdir.glob("*.mp4"), key=lambda item: item.stat().st_mtime)
    if not mp4_files:
        raise FileNotFoundError(f"No video was created in {video_subdir}")

    recorded_episode = dict(solved_episode)
    recorded_episode["states"] = replayed_states
    recorded_episode["video_dir"] = str(video_subdir)
    return recorded_episode, mp4_files[-1], maximum_state_error


print("Recording Original DeePC-Hunt video from its plotted action sequence...")
orig_video_ep, original_video_path, original_replay_error = replay_actions_to_video(
    orig_plot_ep,
    "original_deepc_hunt",
)

print("Recording C Fast-DeePC video from its plotted action sequence...")
c_video_ep, c_video_path, c_replay_error = replay_actions_to_video(
    c_plot_ep,
    "c_fast_deepc_boxqp",
)

print("Original video:", original_video_path)
print("C Fast-DeePC video:", c_video_path)
print("Maximum video/plot state errors:", original_replay_error, c_replay_error)



# Display the generated videos in the notebook.
from IPython.display import Video, display
from pathlib import Path

for label, episode in [("Original DeePC-Hunt", orig_video_ep), ("C Fast-DeePC BoxQP", c_video_ep)]:
    folder = Path(episode["video_dir"])
    mp4_files = sorted(folder.glob("*.mp4"), key=lambda p: p.stat().st_mtime)
    print("\n", label)
    if not mp4_files:
        print("No mp4 file found in", folder)
        continue
    video_path = mp4_files[-1]
    print(video_path)
    display(Video(str(video_path), embed=True, width=520))


## Same-problem timing and closed-loop metrics
The native reduced OSQP problem is initialized once; each control step updates only its linear term.

In [ ]:
def _rocket_c_matrix(pointer, shape):
    return np.ctypeslib.as_array(pointer, shape=(shape[0] * shape[1],)).copy().reshape(shape, order="F")


class RocketReducedNativeOSQP:
    name = "Reduced BoxQP / native OSQP"

    def __init__(self, c_controller):
        self.c = c_controller
        self.H = _rocket_c_matrix(c_controller.data.Hphys, (NZ, NZ))
        self.F = _rocket_c_matrix(c_controller.data.Fphys, (NZ, c_controller.data.nfixed))
        self.solver = osqp.OSQP()
        self.solver.setup(P=sp.csc_matrix(np.triu(self.H)), q=np.zeros(NZ),
                          A=sp.eye(NZ, format="csc"), l=Z_LOWER, u=Z_UPPER,
                          eps_abs=SOLVER_TOL, eps_rel=SOLVER_TOL, max_iter=MAX_SOLVER_ITER, verbose=False)

    def reset(self):
        self.solver.warm_start(x=np.clip(np.zeros(NZ), Z_LOWER, Z_UPPER))

    def solve(self, u_ini, y_ini, y_ref):
        start = time.perf_counter()
        fixed = np.ascontiguousarray(np.r_[u_ini, y_ini], dtype=np.float64)
        tracking = np.r_[-WU @ U_REF, -WY @ np.asarray(y_ref).reshape(-1)]
        q = self.F @ fixed + tracking
        self.solver.update(q=q); answer = self.solver.solve(); elapsed = time.perf_counter() - start
        if answer.info.status_val not in (1, 2):
            raise RuntimeError(f"native OSQP failed: {answer.info.status}")
        z = np.asarray(answer.x).copy(); u, y = z[:NU], z[NU:]
        return {"u": u, "y": y, "g": self.c.recover_g(fixed, u, y),
                "online_time": elapsed, "iterations": int(answer.info.iter),
                "reduced_objective": float(0.5 * z @ self.H @ z + q @ z)}


def _rocket_timing(values):
    values = 1e3 * np.asarray(values, dtype=np.float64)
    return dict(mean_ms=values.mean(), std_ms=values.std(ddof=1) if len(values) > 1 else 0.0,
                p95_ms=np.percentile(values, 95), max_ms=values.max())


def run_rocket_metrics(controller, seed=FIXED_ENV_SEED, initial_position=FIXED_INITIAL_POSITION):
    env, obs, _ = make_environment_at(seed, initial_position)
    landing = np.asarray(env.get_landing_position(), dtype=np.float64); y_ref = reference_from_landing(landing); ref_step = y_ref[:DIM_Y]
    controller.reset(); uw = np.zeros((T_INI, DIM_U)); yw = np.tile(obs[:DIM_Y], (T_INI, 1))
    states, actions, times, iterations, consistency, output_residual, objectives, violations = [], [], [], [], [], [], [], []
    touched_ground = False
    for _ in range(MAX_STEPS):
        y_before = obs[:DIM_Y].copy()
        if (bool(obs[6]) and bool(obs[7])) or touched_ground:
            action = np.zeros(DIM_U); touched_ground = True
        else:
            result = controller.solve(uw.reshape(-1), yw.reshape(-1), y_ref)
            action = result["u"][:DIM_U].copy(); times.append(result["online_time"]); iterations.append(result["iterations"])
            uplan, yplan, g = result["u"], result["y"], result["g"]
            target = np.r_[uw.reshape(-1), yw.reshape(-1), uplan]
            consistency.append(np.linalg.norm(Z @ g - target)); output_residual.append(np.linalg.norm(YF @ g - yplan))
            if "reduced_objective" in result:
                objectives.append(result["reduced_objective"])
            else:
                fixed = np.r_[uw.reshape(-1), yw.reshape(-1)]; H = _rocket_c_matrix(c_controller.data.Hphys, (NZ, NZ)); F = _rocket_c_matrix(c_controller.data.Fphys, (NZ, c_controller.data.nfixed))
                z = np.r_[uplan, yplan]; q = F @ fixed + np.r_[-WU @ U_REF, -WY @ y_ref]
                objectives.append(float(0.5 * z @ H @ z + q @ z))
        obs, _, terminated, truncated, _ = env.step(action); obs = np.asarray(obs, dtype=np.float64)
        states.append(obs[:DIM_Y].copy()); actions.append(action); violations.append(actual_output_violation(obs[:DIM_Y]))
        uw, yw = update_window(uw, yw, action, y_before)
        if terminated or truncated: break
    success = bool(obs[6]) and bool(obs[7]) and obs[1] >= landing[1]
    episode_length = len(actions)
    env.close(); states, actions = np.asarray(states), np.asarray(actions)
    errors = states - ref_step; common_cost = float(sum(e @ Q_STEP @ e + u @ R_STEP @ u for e, u in zip(errors, actions)))
    return dict(states=states, actions=actions, times=np.asarray(times), iterations=np.asarray(iterations),
                consistency=np.asarray(consistency), output_residual=np.asarray(output_residual), objectives=np.asarray(objectives),
                tracking_rmse=float(np.sqrt(np.mean(errors ** 2))), common_cost=common_cost,
                max_actual_violation=float(np.max(violations)),
                final_position_error=float(np.linalg.norm(states[-1, :2] - landing[:2])),
                final_attitude_error=float(abs(states[-1, 4] - landing[2])),
                success=success, episode_length=episode_length, landing=landing.copy())


rocket_native = RocketReducedNativeOSQP(c_controller)
rocket_full = run_rocket_metrics(cvx_controller)
rocket_osqp = run_rocket_metrics(rocket_native)
rocket_c = run_rocket_metrics(c_controller)
orig_states = np.asarray(orig_plot_ep["states"])[1:1 + len(orig_plot_ep["actions"])]
orig_actions = np.asarray(orig_plot_ep["actions"]); orig_ref = reference_from_landing(orig_plot_ep["landing"])[:DIM_Y]
orig_errors = orig_states - orig_ref
rocket_original_metrics = {"tracking_rmse": float(np.sqrt(np.mean(orig_errors ** 2))),
    "common_cost": float(sum(e @ Q_STEP @ e + u @ R_STEP @ u for e, u in zip(orig_errors, orig_actions))),
    "max_actual_violation": float(max(actual_output_violation(y) for y in orig_states)),
    "final_position_error": orig_plot_ep["final_position_error"],
    "final_attitude_error": orig_plot_ep["final_attitude_error"]}
print("\nTIMING (wall-clock online time, ms)")
for name, result in [("Full soft quadratic / OSQP", rocket_full), ("Reduced BoxQP / native OSQP", rocket_osqp), ("Reduced BoxQP / C", rocket_c)]:
    print(name, _rocket_timing(result["times"]), "mean_iter=", float(np.mean(result["iterations"])), "max_iter=", int(np.max(result["iterations"])))
print("\nC PLANT METRICS:", {k: rocket_c[k] for k in ("tracking_rmse", "common_cost", "max_actual_violation", "final_position_error", "final_attitude_error")})
print("ORIGINAL DeePC-HUNT PLANT METRICS:", rocket_original_metrics)
print("C behavioral residuals: consistency mean/max =", rocket_c["consistency"].mean(), rocket_c["consistency"].max())
print("C behavioral residuals: future-output mean/max =", rocket_c["output_residual"].mean(), rocket_c["output_residual"].max())
n = min(len(rocket_osqp["states"]), len(rocket_c["states"])); m = min(len(rocket_osqp["objectives"]), len(rocket_c["objectives"]))
print("Reduced OSQP/C trajectory RMSE =", np.sqrt(np.mean((rocket_osqp["states"][:n] - rocket_c["states"][:n]) ** 2)))
print("Reduced OSQP/C maximum control difference =", np.max(np.abs(rocket_osqp["actions"][:n] - rocket_c["actions"][:n])))
print("Reduced OSQP/C maximum objective difference =", np.max(np.abs(rocket_osqp["objectives"][:m] - rocket_c["objectives"][:m])))


In [ ]:
# Both trajectories come from independent closed-loop simulations with the same environment seed, initial condition, reference, and stopping rule.

full_states = np.asarray(rocket_full["states"], dtype=np.float64)
c_states = np.asarray(rocket_c["states"], dtype=np.float64)
full_actions = np.asarray(rocket_full["actions"], dtype=np.float64)
c_actions = np.asarray(rocket_c["actions"], dtype=np.float64)

common_state_steps = min(len(full_states), len(c_states))
common_action_steps = min(len(full_actions), len(c_actions))

full_c_trajectory_rmse = float(np.sqrt(np.mean(
    (full_states[:common_state_steps] - c_states[:common_state_steps]) ** 2
)))
full_c_max_control_difference = float(np.max(np.abs(
    full_actions[:common_action_steps] - c_actions[:common_action_steps]
)))

claim_reproduced = bool(np.isclose(
    full_c_trajectory_rmse, claimed_rmse, rtol=0.05, atol=1.0e-8
))

print("Original DeePC-Hunt success / steps:",
      orig_plot_ep["success"], orig_plot_ep["episode_length"])
print("Full soft CVXPY success / steps:",
      rocket_full["success"], rocket_full["episode_length"])
print("C Fast-DeePC success / steps:",
      rocket_c["success"], rocket_c["episode_length"])
print("Compared state steps:", common_state_steps)
print("Full CVXPY/C closed-loop trajectory RMSE =",
      full_c_trajectory_rmse)
print("Full CVXPY/C maximum control difference =",
      full_c_max_control_difference)


In [ ]:
times = np.asarray(orig_plot_ep["solve_times"], dtype=np.float64)
iterations = np.asarray(orig_plot_ep["iterations"], dtype=np.float64)

print("Recorded original solves:", len(times))
print("Recorded original iterations:", len(iterations))

if times.size == 0:
    raise RuntimeError(
        "No original DeePC-Hunt solves were recorded. "
        "Rerun the cell that creates selected_video_case."
    )

times_ms = 1e3 * times

print(
    r"& Original DeePC-Hunt"
    f" & {np.mean(times_ms):.1f}"
    f" & {np.std(times_ms, ddof=1):.1f}"
    f" & {np.mean(iterations):.0f}/{int(np.max(iterations))}"
    r" \\"
)